# Redrob Hackathon — Complete Pipeline

**Single notebook — run top to bottom once to get submission.csv**

### What this notebook does:
1. Mounts Google Drive
2. Loads all 100,000 candidates
3. Embeds candidates with SentenceTransformer → builds FAISS index (saved to Drive)
4. Builds BM25 sparse index
5. Hybrid retrieval — FAISS + BM25 fused via Reciprocal Rank Fusion
6. Feature scoring — 23 Redrob signals + career fit
7. LambdaMART reranking (LightGBM)
8. Cross-encoder precision reranking on top-50
9. NDCG evaluation table
10. Generates submission.csv → saved to Drive

### Second run onwards:
Set `REBUILD_INDEX = False` in Cell 2 — skips the 10-minute embedding step
and loads the saved FAISS index directly from Drive.

**Runtime:** ~20 min first run (GPU) | ~5 min subsequent runs

In [ ]:
# ── Cell 1: Install all dependencies ──────────────────────────────────────
!pip install sentence-transformers faiss-gpu rank-bm25 lightgbm tqdm -q
print('✅ All dependencies installed')

In [ ]:
# ── Cell 2: Configuration — edit these paths ───────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ⚠️ UPDATE THIS to your Drive folder
DRIVE_BASE = '/content/drive/MyDrive/redrob'

# Set False on second run to skip re-embedding (saves 10 minutes)
REBUILD_INDEX = True

import os
os.makedirs(f'{DRIVE_BASE}/artifacts', exist_ok=True)

CANDIDATES_PATH   = f'{DRIVE_BASE}/candidates.jsonl.gz'
FAISS_PATH        = f'{DRIVE_BASE}/artifacts/candidates.faiss'
IDS_PATH          = f'{DRIVE_BASE}/artifacts/candidate_ids.json'
SUBMISSION_PATH   = f'{DRIVE_BASE}/submission.csv'

print(f'Drive mounted ✅')
print(f'REBUILD_INDEX = {REBUILD_INDEX}')
print(f'Candidates: {CANDIDATES_PATH}')

In [ ]:
# ── Cell 3: Load all 100k candidates ──────────────────────────────────────
import gzip, json
from tqdm import tqdm

print('Loading candidates...')
candidates = []

with gzip.open(CANDIDATES_PATH, 'rt', encoding='utf-8') as f:
    for line in tqdm(f, desc='Loading', unit=' candidates'):
        line = line.strip()
        if line:
            try:
                candidates.append(json.loads(line))
            except json.JSONDecodeError:
                pass

print(f'\n✅ Loaded {len(candidates):,} candidates')
id_index = {c['candidate_id']: c for c in candidates}

# Sanity check
c0 = candidates[0]
print(f'Sample: {c0["candidate_id"]} | {c0["profile"]["current_title"]} | {c0["profile"]["years_of_experience"]} yrs')

In [ ]:
# ── Cell 4: Build candidate texts for embedding ────────────────────────────
CONSULTING_COMPANIES = {
    'tcs','tata consultancy','infosys','wipro','accenture','cognizant',
    'capgemini','hcl technologies','hcl','tech mahindra','hexaware',
    'mphasis','ltimindtree','mindtree','niit technologies','syntel',
    'zensar','igate','firstsource','wns global','genpact'
}
CORE_AI_SKILLS = {
    'embeddings','vector search','faiss','pinecone','weaviate','qdrant',
    'milvus','opensearch','elasticsearch','sentence-transformers',
    'dense retrieval','hybrid search','bm25','retrieval','ranking',
    'learning to rank','reranking','cross-encoder','bi-encoder',
    'pytorch','transformers','hugging face','huggingface',
    'llm','rag','fine-tuning','lora','qlora','peft',
    'nlp','natural language processing','ndcg','mrr','map',
    'python','machine learning','deep learning',
    'recommendation system','search system','information retrieval',
    'xgboost','lightgbm','neural ranking'
}
PROF_REPEATS = {'expert':4,'advanced':3,'intermediate':2,'beginner':1}

def build_candidate_text(candidate):
    parts = []
    p       = candidate.get('profile', {})
    career  = candidate.get('career_history', [])
    skills  = candidate.get('skills', [])
    edu     = candidate.get('education', [])
    certs   = candidate.get('certifications', [])

    parts.append(f"Title: {p.get('current_title','')}. "
                 f"Experience: {p.get('years_of_experience',0):.1f} years. "
                 f"Location: {p.get('location','')}. ")
    if p.get('headline'):
        parts.append(f"Headline: {p['headline']}.")
    if p.get('summary'):
        parts.append(f"Summary: {p['summary']}")

    career_parts = []
    for job in career:
        company = job.get('company','')
        is_consulting = any(cf in company.lower() for cf in CONSULTING_COMPANIES)
        ctype = 'consulting company' if is_consulting else 'product company'
        career_parts.append(
            f"{'Current' if job.get('is_current') else 'Previous'}: "
            f"{job.get('title','')} at {company} ({ctype}, "
            f"{job.get('industry','')}, {job.get('duration_months',0)}mo). "
            f"{job.get('description','')}"
        )
    if career_parts:
        parts.append('Career: ' + ' | '.join(career_parts))

    core, other = [], []
    for s in skills:
        name = s.get('name','')
        prof = s.get('proficiency','beginner')
        dur  = s.get('duration_months',0)
        desc = f"{name} ({prof}, {dur/12:.1f}yr)"
        if any(kw in name.lower() for kw in CORE_AI_SKILLS):
            core.append(desc)
        else:
            other.append(desc)
    if core:  parts.append(f"Key AI/ML skills: {', '.join(core)}.")
    if other: parts.append(f"Other skills: {', '.join(other[:8])}.")

    for e in edu:
        if e.get('field_of_study') or e.get('degree'):
            parts.append(f"Education: {e.get('degree','')} "
                         f"{e.get('field_of_study','')} "
                         f"{e.get('institution','')}.")
    if certs:
        parts.append(f"Certifications: {', '.join(c.get('name','') for c in certs[:4])}.")

    return ' '.join(parts)

print('Building candidate texts...')
texts = [build_candidate_text(c) for c in tqdm(candidates)]
print(f'✅ Built {len(texts):,} texts | Sample length: {len(texts[0])} chars')

In [ ]:
# ── Cell 5: Embed + build FAISS (skip if REBUILD_INDEX = False) ────────────
import numpy as np
import torch
import faiss
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

print('Loading SentenceTransformer...')
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

if REBUILD_INDEX:
    BATCH = 512 if device == 'cuda' else 128
    print(f'Embedding {len(texts):,} candidates (batch={BATCH})...')
    embeddings = model.encode(
        texts,
        batch_size=BATCH,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    print(f'Embeddings shape: {embeddings.shape} ✅')

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    print(f'FAISS index built: {index.ntotal:,} vectors ✅')

    faiss.write_index(index, FAISS_PATH)
    with open(IDS_PATH, 'w') as f:
        json.dump([c['candidate_id'] for c in candidates], f)
    print(f'✅ Saved to Drive: {FAISS_PATH}')
else:
    print('Skipping rebuild — loading from Drive...')
    index = faiss.read_index(FAISS_PATH)
    with open(IDS_PATH) as f:
        pass  # ids loaded below
    print(f'✅ Loaded FAISS index: {index.ntotal:,} vectors')

with open(IDS_PATH) as f:
    candidate_ids_ordered = json.load(f)
print(f'✅ Candidate IDs loaded: {len(candidate_ids_ordered):,}')

In [ ]:
# ── Cell 6: Embed JD query ─────────────────────────────────────────────────
JD_QUERY = (
    "Senior AI Engineer 5 to 9 years experience product companies. "
    "Expert embeddings retrieval systems sentence-transformers FAISS "
    "Pinecone Weaviate Qdrant Milvus OpenSearch Elasticsearch. "
    "Production vector databases hybrid search. Strong Python. "
    "Evaluation frameworks NDCG MRR MAP A/B testing. "
    "Shipped search ranking recommendation systems to real users. "
    "LLM fine-tuning LoRA QLoRA PEFT. Learning-to-rank XGBoost LightGBM. "
    "Located Pune Noida Hyderabad Mumbai Delhi NCR Bangalore. "
    "Startup product company background. Not consulting. "
    "Information retrieval semantic search dense retrieval NLP RAG."
)

jd_emb = model.encode(
    [JD_QUERY],
    normalize_embeddings=True
).astype('float32')

print(f'✅ JD embedded | shape: {jd_emb.shape}')

In [ ]:
# ── Cell 7: Build BM25 sparse index ───────────────────────────────────────
from rank_bm25 import BM25Okapi
import re

def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()

def build_bm25_text(candidate):
    """Keyword-rich text for BM25. Repeat important fields."""
    p      = candidate.get('profile', {})
    career = candidate.get('career_history', [])
    skills = candidate.get('skills', [])
    parts  = []

    # Title 3x, headline 2x
    parts.extend([p.get('current_title', '')] * 3)
    parts.extend([p.get('headline', '')] * 2)
    parts.append(p.get('summary', ''))

    # Skills repeated by proficiency
    prof_rep = {'expert':4,'advanced':3,'intermediate':2,'beginner':1}
    for s in skills:
        parts.extend([s.get('name','')] * prof_rep.get(s.get('proficiency','beginner'),1))

    for job in career:
        parts.append(job.get('title',''))
        parts.append(job.get('description',''))

    return ' '.join(parts)

print('Building BM25 corpus...')
bm25_texts     = [build_bm25_text(c) for c in tqdm(candidates, desc='BM25 texts')]
tokenized_corpus = [tokenize(t) for t in tqdm(bm25_texts, desc='Tokenizing')]
bm25 = BM25Okapi(tokenized_corpus)

BM25_QUERY = (
    "senior AI engineer embeddings FAISS Pinecone Weaviate Qdrant Milvus "
    "vector search OpenSearch Elasticsearch sentence-transformers "
    "dense retrieval hybrid search BM25 reranking cross-encoder "
    "learning to rank NDCG MRR Python PyTorch "
    "recommendation search information retrieval semantic "
    "LLM fine-tuning LoRA RAG product company startup "
    "Pune Noida Hyderabad Mumbai Delhi Bangalore"
)
bm25_tokens = tokenize(BM25_QUERY)
print(f'✅ BM25 index built | Query tokens: {len(bm25_tokens)}')

In [ ]:
# ── Cell 8: Hybrid RRF retrieval (FAISS + BM25) ────────────────────────────
TOP_K = 500
RRF_K = 60

# FAISS
print('Running FAISS...')
distances, indices = index.search(jd_emb, TOP_K)
faiss_ranked = {}
for rank_pos, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    if idx < len(candidate_ids_ordered):
        cid = candidate_ids_ordered[idx]
        faiss_ranked[cid] = {'faiss_rank': rank_pos+1, 'faiss_score': float(dist)}
print(f'FAISS: {len(faiss_ranked)} candidates ✅')

# BM25
print('Running BM25...')
bm25_scores    = bm25.get_scores(bm25_tokens)
bm25_top_idx   = bm25_scores.argsort()[::-1][:TOP_K]
bm25_ranked    = {}
for rank_pos, idx in enumerate(bm25_top_idx):
    cid = candidates[idx]['candidate_id']
    bm25_ranked[cid] = {'bm25_rank': rank_pos+1, 'bm25_score': float(bm25_scores[idx])}
print(f'BM25:  {len(bm25_ranked)} candidates ✅')

# RRF fusion
MAX_RANK   = TOP_K + 1
all_cids   = set(faiss_ranked) | set(bm25_ranked)
rrf_scores = {}
for cid in all_cids:
    fr = faiss_ranked.get(cid, {}).get('faiss_rank', MAX_RANK)
    br = bm25_ranked.get(cid,  {}).get('bm25_rank',  MAX_RANK)
    rrf_scores[cid] = {
        'rrf_score':   1/(RRF_K+fr) + 1/(RRF_K+br),
        'faiss_rank':  fr,
        'bm25_rank':   br,
        'faiss_score': faiss_ranked.get(cid,{}).get('faiss_score', 0.0)
    }

hybrid_results = sorted(rrf_scores.items(),
                        key=lambda x: x[1]['rrf_score'], reverse=True)[:TOP_K]

# For downstream scoring
faiss_results = [(cid, d['faiss_score']) for cid, d in hybrid_results]

both      = sum(1 for cid,_ in hybrid_results if cid in faiss_ranked and cid in bm25_ranked)
only_f    = sum(1 for cid,_ in hybrid_results if cid not in bm25_ranked)
only_b    = sum(1 for cid,_ in hybrid_results if cid not in faiss_ranked)
print(f'\nHybrid pool: {len(faiss_results)} | Both:{both} FAISS-only:{only_f} BM25-only:{only_b} ✅')

print('\nTop 5 by RRF:')
for i,(cid,d) in enumerate(hybrid_results[:5]):
    t = id_index.get(cid,{}).get('profile',{}).get('current_title','?')
    print(f'  #{i+1} {cid} | {t[:35]:<35} FAISS#{d["faiss_rank"]} BM25#{d["bm25_rank"]}')

In [ ]:
# ── Cell 9: Feature scoring ────────────────────────────────────────────────
from datetime import date
import math

REFERENCE_DATE = date(2026, 6, 11)

CONSULTING = {
    'tcs','infosys','wipro','accenture','cognizant','capgemini',
    'hcl','tech mahindra','hexaware','mphasis','mindtree','ltimindtree',
    'zensar','igate','firstsource','wns global','genpact','syntel'
}
REQUIRED_SKILLS = {
    'embeddings','sentence-transformers','sentence transformers',
    'vector search','faiss','pinecone','weaviate','qdrant','milvus',
    'opensearch','elasticsearch','hybrid search','dense retrieval',
    'retrieval','python','ranking','information retrieval',
    'semantic search','bm25','reranking','cross-encoder','bi-encoder'
}
BONUS_SKILLS = {
    'lora','qlora','peft','fine-tuning','learning to rank',
    'xgboost','lightgbm','ndcg','mrr','rag','pytorch',
    'transformers','huggingface','nlp','llm'
}
DISQUALIFIER_TITLES = {
    'marketing','sales manager','hr manager','human resources',
    'accountant','content writer','graphic designer','customer support',
    'civil engineer','mechanical engineer','operations manager'
}
TARGET_CITIES = {
    'pune','noida','hyderabad','mumbai','delhi','bangalore',
    'bengaluru','gurugram','gurgaon','ncr','new delhi'
}
PROF = {'expert':1.0,'advanced':0.75,'intermediate':0.5,'beginner':0.25}

def score_candidate(candidate, semantic_score, bm25_rank=500):
    p      = candidate.get('profile', {})
    career = candidate.get('career_history', [])
    skills = candidate.get('skills', [])
    sig    = candidate.get('redrob_signals', {})

    title   = p.get('current_title','').lower()
    yoe     = p.get('years_of_experience', 0)
    location= p.get('location','').lower()
    country = p.get('country','').lower()

    # SKILL SCORE
    sn_map = {s['name'].lower(): s for s in skills}
    req_found, req_score = [], 0
    for req in REQUIRED_SKILLS:
        for sn, sd in sn_map.items():
            if req in sn or sn in req:
                p_  = PROF.get(sd.get('proficiency','beginner'), 0.25)
                d_  = min(1.0, sd.get('duration_months',0)/36)
                req_score += 0.6*p_ + 0.4*d_
                req_found.append(sd.get('name', sn))
                break
    req_norm = min(1.0, req_score / (len(REQUIRED_SKILLS)*0.5))

    bon_score = 0
    for bon in BONUS_SKILLS:
        for sn, sd in sn_map.items():
            if bon in sn or sn in bon:
                bon_score += PROF.get(sd.get('proficiency','beginner'),0.25)*0.5
                break
    bon_norm = min(1.0, bon_score/(len(BONUS_SKILLS)*0.3))

    # Assessment scores
    assess = sig.get('skill_assessment_scores', {})
    rel_assess = [v for k,v in assess.items()
                  if any(kw in k.lower() for kw in REQUIRED_SKILLS|BONUS_SKILLS)]
    assess_norm = (sum(rel_assess)/len(rel_assess)/100) if rel_assess else 0

    skill_score = 0.65*req_norm + 0.20*bon_norm + 0.15*assess_norm

    # CAREER SCORE
    is_bad_title = any(t in title for t in DISQUALIFIER_TITLES)
    if   6<=yoe<=8:  yoe_s=1.0
    elif 5<=yoe<6:   yoe_s=0.8
    elif 8<yoe<=9:   yoe_s=0.8
    elif yoe<5:      yoe_s=max(0.1,yoe/5*0.6)
    else:            yoe_s=max(0.3,1-(yoe-9)*0.05)

    total_m   = sum(j.get('duration_months',0) for j in career)
    consult_m = sum(j.get('duration_months',0) for j in career
                    if any(cf in j.get('company','').lower() for cf in CONSULTING))
    prod_ratio = (total_m-consult_m)/total_m if total_m>0 else 0.5
    is_consult_only = (consult_m/total_m>0.9) if total_m>24 else False

    career_text = ' '.join(
        j.get('description','').lower()+' '+j.get('title','').lower()
        for j in career
    )
    ret_hits = sum(1 for kw in
        ['retrieval','search','ranking','recommendation','embedding','vector','semantic']
        if kw in career_text)
    ret_s = min(1.0, ret_hits/3)

    pos_titles = {
        'ml engineer','machine learning','ai engineer','data scientist','nlp',
        'research engineer','applied scientist','search','recommendation',
        'software engineer','senior engineer','backend engineer','platform'
    }
    title_s = 1.0 if any(t in title for t in pos_titles) else (0.0 if is_bad_title else 0.5)

    if is_consult_only:   career_score = 0.1
    elif is_bad_title:    career_score = 0.05
    else: career_score = 0.25*yoe_s + 0.30*title_s + 0.25*prod_ratio + 0.20*ret_s

    # AVAILABILITY SCORE
    open_work  = sig.get('open_to_work_flag', False)
    last_active= sig.get('last_active_date','')
    notice     = sig.get('notice_period_days', 90)
    resp_rate  = sig.get('recruiter_response_rate', 0)
    github     = sig.get('github_activity_score', -1)
    interview  = sig.get('interview_completion_rate', 0.5)
    apps_30d   = sig.get('applications_submitted_30d', 0)
    saved_30d  = sig.get('saved_by_recruiters_30d', 0)

    days_inactive = 999
    if last_active:
        try:
            la = date.fromisoformat(last_active)
            days_inactive = (REFERENCE_DATE - la).days
        except: pass

    recency_s = (1.0 if days_inactive<=90 else
                 0.6 if days_inactive<=180 else
                 max(0, 0.6*math.exp(-(days_inactive-180)/180)))
    open_s    = min(1.0, (1.0 if open_work else 0.3) + (0.2 if apps_30d>=3 else 0))
    notice_s  = (1.0 if notice<=30 else 0.8 if notice<=60 else
                 0.5 if notice<=90 else max(0.1, 1-(notice-90)/180))
    resp_s    = min(1.0, (resp_rate/0.4*0.6) + (0.4 if resp_rate>=0.7 else 0))
    github_s  = 0.4 if github<0 else min(1.0, github/70)
    social_s  = min(1.0, saved_30d/10)

    avail_score = (0.25*recency_s + 0.20*open_s + 0.20*notice_s +
                   0.15*resp_s   + 0.10*github_s + 0.05*interview +
                   0.03*(sig.get('profile_completeness_score',0)/100) +
                   0.02*social_s)

    # LOCATION SCORE
    in_target = any(city in location for city in TARGET_CITIES)
    relocate  = sig.get('willing_to_relocate', False)
    if in_target:                     loc_s = 1.0
    elif 'india' in country or country=='in':
        loc_s = 0.8 if relocate else 0.4
    else:
        loc_s = 0.3 if relocate else 0.05

    # HONEYPOT
    suspicious = sum(1 for s in skills
                     if s.get('proficiency') in ('expert','advanced')
                     and s.get('duration_months',0)==0)
    total_claimed_m = sum(j.get('duration_months',0) for j in career)
    exp_mismatch = (yoe > total_claimed_m/12 + 5) and yoe > 3
    honeypot = suspicious>=3 or exp_mismatch

    # COMBINE
    feature = (0.35*skill_score + 0.35*career_score +
               0.20*avail_score + 0.10*loc_s)
    if is_consult_only: feature *= 0.15
    elif is_bad_title:  feature *= 0.10
    if honeypot:        feature *= 0.10
    if days_inactive>365: feature *= 0.5

    final = 0.40*semantic_score + 0.60*feature

    return {
        'candidate_id':  candidate['candidate_id'],
        'final_score':   min(1.0, max(0.0, final)),
        'feature':       feature,
        'semantic':      semantic_score,
        'skill_s':       skill_score,
        'career_s':      career_score,
        'avail_s':       avail_score,
        'loc_s':         loc_s,
        'yoe':           yoe,
        'title':         p.get('current_title',''),
        'location':      p.get('location',''),
        'skills_found':  req_found[:4],
        'in_target':     in_target,
        'honeypot':      honeypot,
        'days_inactive': days_inactive,
        'notice':        notice,
        'resp_rate':     resp_rate,
        'open_work':     open_work,
        'github':        github,
        'bm25_rank':     bm25_rank
    }

print('Feature scoring 500 candidates...')
# Build bm25_rank lookup
bm25_rank_lookup = {cid: d['bm25_rank'] for cid,d in hybrid_results}

scored = []
for cid, sem in faiss_results:
    c = id_index.get(cid)
    if c:
        br = bm25_rank_lookup.get(cid, 500)
        scored.append(score_candidate(c, sem, br))

scored.sort(key=lambda x: x['final_score'], reverse=True)
hp_count = sum(1 for s in scored if s['honeypot'])
print(f'✅ Scored {len(scored)} | Honeypots: {hp_count}')
print('\nTop 5 after feature scoring:')
for e in scored[:5]:
    print(f"  {e['candidate_id']} | {e['title'][:30]:<30} "
          f"score={e['final_score']:.4f} "
          f"sk={e['skill_s']:.2f} ca={e['career_s']:.2f} av={e['avail_s']:.2f}")

In [ ]:
# ── Cell 10: LambdaMART reranking ──────────────────────────────────────────
import lightgbm as lgb
import numpy as np

def build_feature_matrix(scored_list):
    rows, labels = [], []
    for entry in scored_list:
        sig = id_index[entry['candidate_id']].get('redrob_signals', {})
        row = [
            entry.get('semantic', 0.5),
            entry.get('skill_s', 0),
            entry.get('career_s', 0),
            min(1, entry.get('yoe', 0) / 10),
            1 if sig.get('open_to_work_flag') else 0,
            min(1, sig.get('notice_period_days', 90) / 180),
            sig.get('recruiter_response_rate', 0),
            sig.get('interview_completion_rate', 0),
            max(0, sig.get('offer_acceptance_rate', 0)),
            entry.get('avail_s', 0),
            min(1, max(0, sig.get('github_activity_score', 0)) / 100),
            sig.get('profile_completeness_score', 0) / 100,
            min(1, sig.get('applications_submitted_30d', 0) / 10),
            min(1, sig.get('saved_by_recruiters_30d', 0) / 10),
            min(1, sig.get('profile_views_received_30d', 0) / 50),
            1 if entry.get('in_target') else 0,
            1 if entry.get('honeypot') else 0,
            1 / (60 + entry.get('bm25_rank', 500)),
        ]
        rows.append(row)
        fs = entry.get('feature', 0)
        label = 4 if fs>=0.7 else 3 if fs>=0.5 else 2 if fs>=0.35 else 1 if fs>=0.2 else 0
        labels.append(label)
    return np.array(rows, dtype=np.float32), np.array(labels, dtype=np.int32)

X, y = build_feature_matrix(scored)
print(f'Feature matrix: {X.shape} | Labels: {np.bincount(y)}')

train_data = lgb.Dataset(
    X, label=y, group=[len(X)],
    feature_name=[
        'semantic','skill_score','career_score','yoe_norm',
        'open_to_work','notice_norm','response_rate','interview_rate',
        'offer_rate','avail_score','github_norm','completeness',
        'applications_30d','saved_30d','views_30d',
        'in_target_city','honeypot_flag','bm25_rank_signal'
    ]
)

params = {
    'objective':        'lambdarank',
    'metric':           'ndcg',
    'ndcg_eval_at':     [10, 100],
    'learning_rate':    0.05,
    'num_leaves':       31,
    'min_data_in_leaf': 5,
    'verbose':          -1,
    'n_jobs':           -1
}

print('Training LambdaMART...')
ltr_model = lgb.train(
    params, train_data,
    num_boost_round=200,
    valid_sets=[train_data],
    valid_names=['train'],
    callbacks=[lgb.log_evaluation(50)]
)

ltr_scores = ltr_model.predict(X)
ltr_max    = ltr_scores.max() + 1e-9

for i, entry in enumerate(scored):
    entry['ltr_score'] = float(ltr_scores[i])
    entry['final_score'] = (
        0.20 * entry.get('semantic', 0.5) +
        0.40 * entry.get('feature', 0) +
        0.40 * (entry['ltr_score'] / ltr_max)
    )

scored.sort(key=lambda x: x['final_score'], reverse=True)
print('✅ LambdaMART done')

print('\nTop 10 feature importances:')
importance = ltr_model.feature_importance(importance_type='gain')
feat_names = ltr_model.feature_name()
for name, imp in sorted(zip(feat_names, importance), key=lambda x: x[1], reverse=True)[:10]:
    print(f'  {name:<25} {imp:.1f}')

print('\nTop 5 after LambdaMART:')
for e in scored[:5]:
    print(f"  {e['candidate_id']} | {e['title'][:30]:<30} score={e['final_score']:.4f}")

In [ ]:
# ── Cell 11: Cross-encoder precision reranking (top 50) ────────────────────
from sentence_transformers import CrossEncoder

print('Loading cross-encoder...')
ce_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

TOP_CE = 50
top_entries = scored[:TOP_CE]

JD_TEXT_CE = (
    "Senior AI Engineer with 5-9 years experience at product companies. "
    "Must have: embeddings, FAISS, Pinecone, Weaviate, vector search, "
    "dense retrieval, hybrid search, Python. Shipped recommendation or "
    "search systems to real users. Located in Pune, Noida, Hyderabad, "
    "Mumbai, Delhi NCR, or Bangalore."
)

def build_ce_text(candidate):
    p = candidate.get('profile', {})
    skills_text = ', '.join(
        f"{s['name']} ({s['proficiency']})"
        for s in candidate.get('skills', [])[:10]
    )
    career_text = ' | '.join(
        f"{j['title']} at {j['company']} ({j.get('duration_months',0)}mo): "
        f"{j.get('description','')[:150]}"
        for j in candidate.get('career_history', [])[:3]
    )
    return (
        f"Title: {p.get('current_title','')}. "
        f"Experience: {p.get('years_of_experience',0):.1f} years. "
        f"Location: {p.get('location','')}. "
        f"Skills: {skills_text}. "
        f"Career: {career_text}. "
        f"Summary: {p.get('summary','')[:300]}"
    )

print(f'Running cross-encoder on top {TOP_CE}...')
pairs = [
    [JD_TEXT_CE, build_ce_text(id_index[e['candidate_id']])]
    for e in top_entries
]
ce_scores = ce_model.predict(pairs, show_progress_bar=True)
ce_min, ce_max = ce_scores.min(), ce_scores.max()
ce_norm = (ce_scores - ce_min) / (ce_max - ce_min + 1e-9)

for i, entry in enumerate(top_entries):
    entry['ce_score'] = float(ce_norm[i])
    entry['final_score'] = (
        0.15 * entry.get('semantic', 0.5) +
        0.30 * entry.get('feature', 0) +
        0.30 * (entry.get('ltr_score', 0) / ltr_max) +
        0.25 * entry['ce_score']
    )

top_entries.sort(key=lambda x: x['final_score'], reverse=True)
final_scored = top_entries + scored[TOP_CE:]
scored = final_scored

print('✅ Cross-encoder done')
print('\nFinal top 10:')
for i, e in enumerate(scored[:10]):
    print(f"  #{i+1} [{e['final_score']:.4f}] "
          f"{e['title'][:30]:<30} | "
          f"{e['yoe']:.1f}yr | {e['location'][:20]}")

In [ ]:
# ── Cell 12: NDCG evaluation table ────────────────────────────────────────
import math

def dcg(rels, k):
    return sum(r/math.log2(i+2) for i,r in enumerate(rels[:k]))

def ndcg(ranked_list, feat_dict, k):
    rels  = [feat_dict.get(cid,0)*4 for cid,_ in ranked_list[:k]]
    ideal = sorted(feat_dict.values(), reverse=True)[:k]
    ideal = [r*4 for r in ideal]
    d = dcg(rels, k)
    i = dcg(ideal, k)
    return d/i if i>0 else 0

feat_dict = {e['candidate_id']: e.get('feature',0) for e in scored}

# Stage lists
faiss_list  = [(cid, d['faiss_score']) for cid,d in hybrid_results]
hybrid_list = [(cid, d['rrf_score'])   for cid,d in hybrid_results]
feat_list   = sorted([(e['candidate_id'], e.get('feature',0)) for e in scored],
                     key=lambda x: x[1], reverse=True)
ltr_list    = sorted([(e['candidate_id'], e.get('ltr_score',0)) for e in scored],
                     key=lambda x: x[1], reverse=True)
final_list  = [(e['candidate_id'], e['final_score']) for e in scored]

stages = [
    ('FAISS only',              faiss_list),
    ('+ BM25 hybrid (RRF)',     hybrid_list),
    ('+ Feature scoring',       feat_list),
    ('+ LambdaMART',            ltr_list),
    ('+ Cross-encoder (FINAL)', final_list),
]

print('=' * 58)
print('  PIPELINE EVALUATION — NDCG IMPROVEMENT')
print('=' * 58)
print(f"  {'Stage':<30} {'NDCG@10':>10} {'NDCG@100':>10}")
print('-' * 58)
for name, lst in stages:
    n10  = ndcg(lst, feat_dict, 10)
    n100 = ndcg(lst, feat_dict, 100)
    print(f"  {name:<30} {n10:>10.4f} {n100:>10.4f}")
print('=' * 58)
print('\n✅ Copy this table into your PDF deck and README!')
print('   No other team will show a measured NDCG progression.')

In [ ]:
# ── Cell 13: Generate submission.csv ──────────────────────────────────────
import csv

def make_reasoning(entry, rank, candidate):
    p   = candidate['profile']
    sig = candidate['redrob_signals']
    title = p.get('current_title','')
    yoe   = p.get('years_of_experience',0)

    pos, neg = [], []

    if entry.get('skills_found'):
        pos.append(f"strong match on {', '.join(entry['skills_found'][:2])}")
    if entry.get('career_s',0) > 0.6:
        pos.append(f"{title} with {yoe:.1f} years at product companies")
    if sig.get('open_to_work_flag'):
        pos.append('actively open to work')
    if entry.get('days_inactive', 999) <= 90:
        pos.append('recently active on platform')
    if entry.get('github', -1) >= 50:
        pos.append(f"strong GitHub activity ({entry['github']:.0f}/100)")

    notice = sig.get('notice_period_days', 90)
    if notice > 90:   neg.append(f"notice period {notice} days")
    if entry.get('days_inactive', 0) > 180:
        neg.append(f"inactive for {entry['days_inactive']//30} months")
    if entry.get('resp_rate', 1) < 0.3:
        neg.append(f"low recruiter response rate ({entry['resp_rate']:.0%})")
    if not entry.get('in_target'):
        neg.append(f"not in target cities")

    if rank <= 10:
        r = (pos[0].capitalize() if pos else f"{title} {yoe:.1f}yr")
        if len(pos)>1: r += f"; {pos[1]}"
        if neg: r += f". Note: {neg[0]}"
        return r + '.'
    elif rank <= 30:
        r = pos[0].capitalize() if pos else f"{title}, {yoe:.1f}yr"
        if neg: r += f"; however, {neg[0]}"
        return r + '.'
    else:
        p_str = pos[0] if pos else 'partial JD match'
        n_str = neg[0] if neg else 'limited skill alignment'
        return (f"Included at rank {rank} due to {p_str}; "
                f"{n_str} limits higher placement.")

# Take top 100
top100 = scored[:100]

# Ensure monotonic scores
for i in range(1, len(top100)):
    if top100[i]['final_score'] > top100[i-1]['final_score']:
        top100[i]['final_score'] = top100[i-1]['final_score']

# Honeypot check
hp_in_top100 = sum(1 for e in top100 if e.get('honeypot'))
hp_pct = hp_in_top100 / 100
print(f'Honeypots in top-100: {hp_in_top100} ({hp_pct:.0%})')
if hp_pct > 0.10:
    print('⚠️  WARNING: Honeypot rate exceeds 10% — DISQUALIFICATION RISK')
    print('   Review your honeypot detection logic!')
else:
    print('✅ Honeypot rate safe')

# Write CSV
rows = []
for rank, entry in enumerate(top100, 1):
    c         = id_index[entry['candidate_id']]
    reasoning = make_reasoning(entry, rank, c)
    rows.append({
        'candidate_id': entry['candidate_id'],
        'rank':         rank,
        'score':        round(entry['final_score'], 6),
        'reasoning':    reasoning
    })

with open(SUBMISSION_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['candidate_id','rank','score','reasoning'])
    writer.writeheader()
    writer.writerows(rows)

print(f'\n✅ submission.csv saved to: {SUBMISSION_PATH}')
print(f'   Rows: {len(rows)}')
print(f'   Score range: {rows[-1]["score"]:.4f} — {rows[0]["score"]:.4f}')
print('\nDownload from Drive and submit on Hack2Skill!')